# CoreX V1 - Baseline OmniAnomaly Training on Kaggle
**Multivariate Time-Series Anomaly Detection on RobotArm using VAE + Normalizing Flows.**

Settings Required:
- GPU: T4 GPU enabled

In [ ]:
%%bash
# === CELL 1: Copy Project files & Setup Data ===
cd /kaggle/working
rm -rf project

# 1. Find the baseline folder automatically in Kaggle input
SRC=$(find /kaggle/input -type d -name "Version_1_Baseline" | head -1)

if [ -n "$SRC" ]; then
    cp -r "$SRC" project
    echo "✅ Version 1 Baseline copied from: $SRC"
    
    # 2. Copy the all_data.csv to the correct folder inside project
    DATA_SRC=$(find /kaggle/input -name "all_data.csv" -o -name "all-data.csv" | head -1)
    if [ -n "$DATA_SRC" ]; then
        mkdir -p project/data/RobotArm
        cp "$DATA_SRC" project/data/RobotArm/all_data.csv
        echo "✅ Dataset copied to project/data/RobotArm/all_data.csv from: $DATA_SRC"
    else
        echo "⚠️ Warning: Could not find all_data.csv in /kaggle/input. Please upload it!"
    fi
    
    ls project/
else
    echo "❌ Could not find Version_1_Baseline folder. Listing input:"
    find /kaggle/input -maxdepth 3 -type d
fi

In [ ]:
%%bash
# === CELL 2: Environment Setup (Python 3.6 + TF 1.15) ===
set -e

# 1. Install Miniconda if not present
if ! command -v /opt/conda/bin/conda &> /dev/null; then
    echo "=== Installing Miniconda ==="
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /opt/conda -f 2>/dev/null
fi
export PATH=/opt/conda/bin:$PATH

# 2. Accept Conda Terms of Service (TOS)
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
conda config --set always_yes true

# 3. Create conda environment (Python 3.6)
echo "=== Creating corex_env ==="
conda env remove -n corex_env 2>/dev/null || true
conda create -n corex_env python=3.6 -y -q

# 4. Install TensorFlow 1.15 + CUDA
echo "=== Installing TF 1.15 + CUDA ==="
conda install -n corex_env -c defaults tensorflow-gpu=1.15 cudatoolkit=10.0 cudnn=7 numpy=1.16 scipy pandas scikit-learn matplotlib -y -q

# 5. Install specialized dependencies (Using exact matching commit hash from local working requirements)
PIP=/opt/conda/envs/corex_env/bin/pip
echo "=== Installing pip dependencies ==="
$PIP install -q \
    git+https://github.com/haowen-xu/tfsnippet.git@63adaf04d2ffff8dec299623627d55d4bacac598 \
    git+https://github.com/thu-ml/zhusuan.git \
    seaborn \
    imageio

echo ""
echo "=== Verification ==="
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -c "
import tfsnippet; print('tfsnippet: OK')
import zhusuan; print('zhusuan: OK')
"
echo "✅ Environment Fully Configured!"


In [ ]:
%%bash
# === CELL 3: Verify GPU ===
export PATH=/opt/conda/envs/corex_env/bin:$PATH
/opt/conda/envs/corex_env/bin/python -c "
import tensorflow as tf
print('TF Version:', tf.__version__)
print('GPU Available:', tf.test.is_gpu_available())
print('GPU Device Name:', tf.test.gpu_device_name())
"

In [ ]:
%%bash
# === CELL 4: Install Extra Dependencies ===
/opt/conda/envs/corex_env/bin/pip install \
    tensorflow-probability==0.8.0 \
    tqdm==4.28.1 \
    fs==2.3.0 \
    click==7.0 \
    PyYAML==5.4.1 \
    xlrd==2.0.2 \
    openpyxl==3.1.3

In [ ]:
# === CELL 5: Patch main.py & plot_results.py Config for Kaggle Running ===
# Configure number of epochs (User requested 25 epochs)
EPOCHS = 25
import os, re
os.chdir('/kaggle/working/project')

# Patch main.py
with open('main.py', 'r') as f:
    content = f.read()

content = re.sub(r'max_epoch\s*=\s*\d+', f'max_epoch = {EPOCHS}', content)
content = re.sub(r"'max_epoch':\s*\d+", f"'max_epoch': {EPOCHS}", content)
content = re.sub(r'batch_size\s*=\s*\d+', 'batch_size = 25', content)
content = re.sub(r"restore_dir\s*=\s*'[^']*'", 'restore_dir = None', content)

with open('main.py', 'w') as f:
    f.write(content)

print(f'✅ main.py patched: {EPOCHS} epochs, batch_size=25, fresh training')

# Patch plot_results.py paths to match project layout
if os.path.exists('plot_results.py'):
    with open('plot_results.py', 'r') as f:
        plot_content = f.read()

    plot_content = plot_content.replace("result_path = 'result/test_score.pkl'", "result_path = 'results/RobotArm_coreX_v1/test_score.pkl'")
    plot_content = plot_content.replace("label_path = f'processed/{dataset}_test_label.pkl'", "label_path = f'data/processed/{dataset}_test_label.pkl'")

    with open('plot_results.py', 'w') as f:
        f.write(plot_content)
    print('✅ plot_results.py patched to use correct relative data paths')

In [ ]:
%%bash
# === CELL 6: Unpack & Preprocess Data ===
cd /kaggle/working/project
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -u data_preprocess.py

In [ ]:
%%bash
# === CELL 7: Train the Model ===
cd /kaggle/working/project
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -u main.py

In [ ]:
%%bash
# === CELL 8: Generate Baseline Performance Plot ===
cd /kaggle/working/project
if [ -f "plot_results.py" ]; then
    MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -u plot_results.py
fi

In [ ]:
# === CELL 9: Export Results & Checkpoints ===
import zipfile, os, glob

os.chdir('/kaggle/working/project')

def make_zip(name, patterns):
    with zipfile.ZipFile(f'/kaggle/working/{name}', 'w', zipfile.ZIP_DEFLATED) as z:
        for p in patterns:
            for f in glob.glob(p, recursive=True):
                z.write(f)
    print(f'✅ {name} created')

make_zip('coreX_v1_baseline_results.zip', ['results/**/*'])
make_zip('coreX_v1_baseline_model.zip', ['model_coreX_v1/**/*'])

print('\n🎉 Zipped! You can download results from the Output tab.')